In [1]:
""""
Summarize fuel treatment acres within fire perimeters
Maxwell.Cook@colostate.edu
"""

import os, sys

# import the __functions.py (custom functions)
sys.path.append(os.getcwd()) # add code folder to system path
from __functions import *  # imports all custom functions

# local data directories
datadir = '/Users/mcc/Library/CloudStorage/Box-Box/MCC/data/'
projdir = os.path.dirname(os.getcwd())
print(projdir)

print("Ready !")

/Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation
Ready !


In [3]:
# load the incidents / perimeters
incis = os.path.join(projdir, 'data/spatial/mod/ics209plus_2014to2023_perimeters_tcc.gpkg')
incis = gpd.read_file(incis)

# --- Keep only forested incidents
incis = incis.copy()
incis = incis[
    (incis['FUEL_MODEL'].str.contains('Timber', case=False)) |
    (incis['tcc']>=10) # keep forested incidents
]

print(f"Processing for {len(incis)} fires.")
print(f"CRS: {incis.crs}")

# clean the dataframe just a bit for processing
incis = incis[['INCIDENT_ID','INCIDENT_NAME','DISCOVERY_DATE','FINAL_ACRES','geometry']]

Processing for 4259 fires.
CRS: EPSG:5070


In [4]:
# load the fuel treatment data
fp = os.path.join(projdir, 'data/spatial/treatments/twig_treatments_1km.gpkg')
trts = gpd.read_file(fp)
# convert date fields and extract the year
trts['treatment_date'] = pd.to_datetime(trts['treatment_date'], unit='ms', errors='coerce')
trts['actual_completion_date'] = pd.to_datetime(trts['actual_completion_date'], unit='ms', errors='coerce')
trts['year_comp'] = trts['actual_completion_date'].dt.year
# check on the structure
print(f"Columns:\n{trts.columns}")
print(f"CRS: {trts.crs}")
trts[['INCIDENT_ID', 'unique_id', 'treatment_date', 'actual_completion_date', 'year_comp', 'type', 'acres']].head()

Columns:
Index(['activity', 'total_cost', 'type', 'cost_per_uom', 'error', 'fund_code',
       'date_source', 'uom', 'identifier_database', 'fund_source', 'state',
       'SHAPE__Area', 'unique_id', 'actual_completion_date', 'method',
       'agency', 'date_current', 'twig_category', 'equipment', 'activity_code',
       'treatment_date', 'name', 'acres', 'category', 'SHAPE__Length',
       'objectid', 'index_right', 'INCIDENT_ID', 'geometry', 'year_comp'],
      dtype='object')
CRS: EPSG:3857


,INCIDENT_ID,unique_id,treatment_date,actual_completion_date,year_comp,type,acres
0,2019_10665633_SWAN LAKE,109464-6314611,2018-09-16,2018-09-16,2018.0,Thinning,4.713520
1,2019_10665633_SWAN LAKE,112113-6315464,2019-03-04,2019-03-04,2019.0,Mastication,21.531993
2,2019_10665633_SWAN LAKE,112252-6318237,2018-11-19,2018-11-19,2018.0,Hand Pile Burn,1.173396
3,2019_10665633_SWAN LAKE,115263-6311191,2018-09-24,2018-09-24,2018.0,Hand Pile Burn,30.485169
4,2019_10665633_SWAN LAKE,109465-6314611,2018-09-16,2018-09-16,2018.0,Thinning,1.401910


In [5]:
trts['type'].unique()

array(['Thinning', 'Mastication', 'Hand Pile Burn', 'Hand Pile',
       'Fire Use', 'Broadcast Burn', 'Lop and Scatter', None,
       'Biomass Removal', 'Chemical', 'Machine Pile', 'N/A', 'Chipping',
       'Chemical Treatment', 'Grazing', 'Machine Pile Burn', 'Crushing',
       'Mastication/Mowing', 'Jackpot Burn', 'Mowing', 'Biological',
       'Seeding', 'Preparation'], dtype=object)

In [6]:
trts['twig_category'].unique()

array(['Mechanical', 'Planned Ignition', 'Unplanned Ignition', 'Chemical',
       'Biological', None], dtype=object)

In [7]:
# subset to our treatments of interest
trts_keep = [
    'Thinning', # this includes both hand and mechanical thinning
    'Machine Pile', 'Hand Pile',
    'Mastication', 'Lop and Scatter',
    'Broadcast Burn', 'Jackpot Burn', 'Fire Use',
    'Machine Pile Burn', 'Hand Pile Burn',
    'Biomass Removal'
]

# filter our dissolved data
trts_ = trts[trts['type'].isin(trts_keep)].copy()
print(trts_['type'].unique())

['Thinning' 'Mastication' 'Hand Pile Burn' 'Hand Pile' 'Fire Use'
 'Broadcast Burn' 'Lop and Scatter' 'Biomass Removal' 'Machine Pile'
 'Machine Pile Burn' 'Jackpot Burn']


In [13]:
# Classify thinning as either manual or mechanical to match the forest tracker
def classify_thin(row):
    """
    Categorizes forest thinning treatments into Manual or Mechanical
    :param row:
    :return:
    """

    ttype = str(row.get("type") or "").strip().lower()
    twig_type = str(row.get("twig_type") or "").strip().lower()
    activity = str(row.get("activity") or "").strip().lower()

    # is this a thinning treatment??
    is_thin = False
    # a. direct attribution with TWIG type
    is_thin = (ttype == "thinning") # based on treatment type attribute
    # b. thin keyword in activity code
    if "thin" in activity:
        is_thin = True
    # c. including "cut" activities
    thin_cut = [
        'sanitation cut','improvement cut','group selection cut',
        'overstory removal cut','salvage cut','recreation removal'
    ]
    if any(k in activity for k in thin_cut) and 'clearcut' not in activity:
        is_thin = True

    # if not thinning, ignore
    if not is_thin:
        return np.nan

    # --- Manual vs. Mechanical logic

    # use the equipment and method columns
    eq  = str(row.get("equipment") or "").strip().lower()
    mth = str(row.get("method") or "").strip().lower()

    # Equipment-based rules (highest priority)
    manual_eq = {
        "chain saw", "hand work", "hand saw", "manual logging"
    }
    mech_eq = {
        "feller buncher", "tree shear", "dozer", "masticator",
        "rubber tired skidder logging", "helicopter logging -medium",
        "helicopter logging -small",
    }

    if eq in manual_eq:
        return "Manual Thin"
    if eq in mech_eq:
        return "Mechanical Thin"

    # Method-based rules (if equipment is missing/ambiguous)
    manual_m = {
        "manual", "power hand", "manual logging", "cut trees and brush",
    }

    mech_m = {
        "mechanical", "tractor logging", "logging methods",
        "helicopter", "removal",
    }

    if mth in manual_m:
        return "Manual Thin"
    if mth in mech_m:
        return "Mechanical Thin"

    # Default to mechanical
    return "Mechanical Thin"

trts_th = trts_.copy() # work on a copy

# --- Remap thinning classes
trts_th["ACTIVITY"] = trts_th.apply(classify_thin, axis=1)
trts_th["ACTIVITY"] = trts_th["ACTIVITY"].fillna(trts_th["type"])
print(f"Treatment types: {trts_th['ACTIVITY'].unique()}")

# --- Identify potential missingness using the "TWIG Category"
mech_mask = trts_th["type"].isna() & (trts_th["twig_category"].str.lower() == "mechanical")
trts_th.loc[mech_mask, "ACTIVITY"] = "Mechanical Thin"

# for thinning rows that are still NA, default to Mechanical
trts_th.loc[(trts_th["type"] == "Thinning") & (trts_th["type"].isna()), "ACTIVITY"] = "Mechanical Thin"
trts_th['ACTIVITY'] = trts_th['ACTIVITY'].fillna(trts_th['type']) # all others keep their type
print(f"\t\nUpdated treatment types: {trts_th['type'].unique()}")
print(f"Value counts:\n{trts_th['ACTIVITY'].value_counts()}")

Treatment types: ['Mechanical Thin' 'Mastication' 'Hand Pile Burn' 'Hand Pile' 'Fire Use'
 'Broadcast Burn' 'Lop and Scatter' 'Biomass Removal' 'Machine Pile'
 'Manual Thin' 'Machine Pile Burn' 'Jackpot Burn']
	
Updated treatment types: ['Thinning' 'Mastication' 'Hand Pile Burn' 'Hand Pile' 'Fire Use'
 'Broadcast Burn' 'Lop and Scatter' 'Biomass Removal' 'Machine Pile'
 'Machine Pile Burn' 'Jackpot Burn']
Value counts:
ACTIVITY
Mechanical Thin      13121
Manual Thin           9712
Broadcast Burn        8040
Machine Pile          7886
Machine Pile Burn     7639
Biomass Removal       5001
Lop and Scatter       2736
Fire Use              1480
Hand Pile Burn         400
Hand Pile              349
Mastication            223
Jackpot Burn           164
Name: count, dtype: int64


In [14]:
# crop treatments to the fire perimeters
# buffer perimeters 1km
print("Buffering fire perimeters ...")
buffer = incis.copy().to_crs(trts.crs) # match CRS to treatments
buffer['geometry'] = buffer.geometry.buffer(1000)
buffer['fire_buffer_acres'] = buffer.geometry.area / 4046.86 # recalculate mtbs acres with buffer
buffer = buffer.to_crs(trts.crs) # now match the treatment CRS

# loop through fires, crop treatment data to bounds
trt_clips = [] # to store the results
for idx, fire_row in tqdm(buffer.iterrows(), total=buffer.shape[0], desc='Processing Fires'):
    fire_id = fire_row['INCIDENT_ID']
    fire_acres = fire_row['fire_buffer_acres']
    fire_geom = fire_row['geometry']

    # select overlapping treatments
    trts_fire = trts_th[trts_th['INCIDENT_ID'] == fire_id]

    if not trts_fire.empty:
        # Clip treatments to the fire perimeter buffer
        clipped = gpd.clip(trts_fire, fire_geom)
        clipped['fire_buffer_acres'] = fire_acres
        trt_clips.append(clipped)
        del clipped, trts_fire

# Combine all clipped results into one GeoDataFrame
trts_fire = gpd.GeoDataFrame(pd.concat(trt_clips, ignore_index=True), crs=trts.crs)
del trt_clips

# Recalculate area
trts_fire['gis_acres'] = trts_fire.geometry.area / 4046.86
trts_fire.head()

Buffering fire perimeters ...


Processing Fires:   0%|          | 0/4259 [00:00<?, ?it/s]

,activity,total_cost,type,cost_per_uom,error,fund_code,date_source,uom,identifier_database,fund_source,...,category,SHAPE__Length,objectid,index_right,INCIDENT_ID,geometry,year_comp,ACTIVITY,fire_buffer_acres,gis_acres
0,None,NaN,Thinning,NaN,None,None,act_comp_dt,None,NFPORS,No Funding Code,...,Mechanical,2331.101852,758821,29,2019_10665633_SWAN LAKE,"POLYGON ((-16754921.211 8526925.688, -16754921...",2018.0,Mechanical Thin,791697.501455,5.785666
1,None,NaN,Hand Pile Burn,NaN,None,None,act_comp_dt,None,NFPORS,No Funding Code,...,Fire,39037.646712,755364,29,2019_10665633_SWAN LAKE,"MULTIPOLYGON (((-16735966.982 8538331.219, -16...",2018.0,Hand Pile Burn,791697.501455,126.154264
2,None,NaN,Thinning,NaN,None,None,act_comp_dt,None,NFPORS,No Funding Code,...,Mechanical,39037.646712,759993,29,2019_10665633_SWAN LAKE,"MULTIPOLYGON (((-16735966.982 8538331.219, -16...",2018.0,Mechanical Thin,791697.501455,126.154264
3,None,NaN,Hand Pile,NaN,None,None,act_comp_dt,None,NFPORS,No Funding Code,...,Mechanical,39037.646712,759994,29,2019_10665633_SWAN LAKE,"MULTIPOLYGON (((-16735966.982 8538331.219, -16...",2018.0,Hand Pile,791697.501455,126.154264
4,None,NaN,Thinning,NaN,None,None,act_comp_dt,None,NFPORS,No Funding Code,...,Mechanical,29009.742655,766821,29,2019_10665633_SWAN LAKE,"MULTIPOLYGON (((-16745544.633 8532186.314, -16...",2019.0,Mechanical Thin,791697.501455,100.907771


In [15]:
# Flatten (dissolve) by Fire ID, Treatment Type, and Year Completed
flat = trts_fire[trts_fire['ACTIVITY'] != 'None'].dissolve(by=['INCIDENT_ID','ACTIVITY','year_comp']).reset_index()
flat['ACTIVITY'].unique() # check what treatment types we're working with
flat['trt_gis_acres'] = flat.geometry.area / 4046.86 # recalculate the GIS acres
# keep only a few attributes ...
flat_c = flat[['INCIDENT_ID', 'ACTIVITY', 'year_comp', 'trt_gis_acres', 'fire_buffer_acres', 'geometry']].copy()
print(flat_c['ACTIVITY'].unique())

['Broadcast Burn' 'Biomass Removal' 'Hand Pile' 'Mechanical Thin'
 'Machine Pile' 'Machine Pile Burn' 'Manual Thin' 'Fire Use'
 'Lop and Scatter' 'Hand Pile Burn' 'Mastication' 'Jackpot Burn']


In [16]:
# now create the single dissolved layer by fire ID
# add the treatment years as a list
flat_c_fire = (
    flat_c
    .dissolve(
        by=["INCIDENT_ID", "ACTIVITY"], # fire/treatment type
        aggfunc={
            "year_comp": lambda x: sorted(set(x.dropna().astype(int))), # list of treatment years
            "fire_buffer_acres": "first"   # keep the fire acres as a column in the output
        }
    )
    .reset_index()
)

# clip again to fire buffers
flat_c_fire = gpd.clip(flat_c_fire, buffer)
# recalculate the treatment acres
flat_c_fire['treatment_acres'] = flat_c_fire.geometry.area / 4046.86
# calculate the percent treated by treatment type
flat_c_fire['fire_pct_treated'] = (flat_c_fire['treatment_acres'] / flat_c_fire['fire_buffer_acres']) * 100
print(len(flat_c_fire))
flat_c_fire.head()

4112


,INCIDENT_ID,ACTIVITY,geometry,year_comp,fire_buffer_acres,treatment_acres,fire_pct_treated
995,2016_4446992_LONG PINE KEY WF,Broadcast Burn,"MULTIPOLYGON (((-8982259.977 2921000.34, -8982...","[2006, 2007, 2008, 2010, 2012, 2014, 2016]",10502.352417,4543.500809,43.261744
996,2016_4446992_LONG PINE KEY WF,Lop and Scatter,"MULTIPOLYGON (((-8979159.952 2923638.232, -897...",[2016],10502.352417,952.434657,9.068774
3808,2023_15917059_CANEPATCH,Broadcast Burn,"POLYGON ((-9008132.159 2928259.785, -9007874.6...","[2018, 2023]",2614.359916,2589.982905,99.067572
3994,2023_16033325_SANDY,Hand Pile Burn,"POLYGON ((-9020559.372 2982053.985, -9020556.9...",[2019],36493.227277,1.237750,0.003392
3993,2023_16033325_SANDY,Broadcast Burn,"MULTIPOLYGON (((-9020244.295 2981168.668, -902...","[2013, 2014, 2015, 2016, 2017, 2018, 2019]",36493.227277,14494.542079,39.718444


In [17]:
# Join in the unique number of treatments by type
# Count the unique treatments of each type
n_trts = (
    trts_fire
    .groupby(["INCIDENT_ID", "ACTIVITY"])
    .agg(n_treatments=("year_comp", "nunique"))
    .reset_index()
)
# join to the dataframe
flat_c_fire_ = pd.merge(flat_c_fire, n_trts, on=["INCIDENT_ID", "ACTIVITY"], how="left")
# tidy up some of the columns
flat_c_fire_ = flat_c_fire_[[
    'INCIDENT_ID', 'ACTIVITY', 'treatment_acres',
    'fire_buffer_acres', 'fire_pct_treated',
    'year_comp', 'n_treatments', 'geometry'
]]
flat_c_fire_.rename(columns={
    'ACTIVITY': 'treatment_type',
    'year_comp': 'treatments_years'
}, inplace=True)
print(flat_c_fire_['fire_pct_treated'].describe())
flat_c_fire_.head()

count    4112.000000
mean        8.587190
std        17.791205
min         0.000024
25%         0.282127
50%         1.433629
75%         6.791690
max       100.000000
Name: fire_pct_treated, dtype: float64


,INCIDENT_ID,treatment_type,treatment_acres,fire_buffer_acres,fire_pct_treated,treatments_years,n_treatments,geometry
0,2016_4446992_LONG PINE KEY WF,Broadcast Burn,4543.500809,10502.352417,43.261744,"[2006, 2007, 2008, 2010, 2012, 2014, 2016]",7,"MULTIPOLYGON (((-8982259.977 2921000.34, -8982..."
1,2016_4446992_LONG PINE KEY WF,Lop and Scatter,952.434657,10502.352417,9.068774,[2016],1,"MULTIPOLYGON (((-8979159.952 2923638.232, -897..."
2,2023_15917059_CANEPATCH,Broadcast Burn,2589.982905,2614.359916,99.067572,"[2018, 2023]",2,"POLYGON ((-9008132.159 2928259.785, -9007874.6..."
3,2023_16033325_SANDY,Hand Pile Burn,1.237750,36493.227277,0.003392,[2019],1,"POLYGON ((-9020559.372 2982053.985, -9020556.9..."
4,2023_16033325_SANDY,Broadcast Burn,14494.542079,36493.227277,39.718444,"[2013, 2014, 2015, 2016, 2017, 2018, 2019]",7,"MULTIPOLYGON (((-9020244.295 2981168.668, -902..."


In [18]:
# Identify "Thin+Rx Fire" category
thin_cats = ['Manual Thin', 'Mechanical Thin']
thin = flat_c_fire_[flat_c_fire_['treatment_type'].isin(thin_cats)]
rx = flat_c_fire_[flat_c_fire_['treatment_type']=="Broadcast Burn"]

# Perform an intersection of these treatment types
thin_rx = gpd.overlay(thin, rx, how="intersection", keep_geom_type=False)
# make sure we are working within the same fire event
thin_rx = thin_rx[thin_rx['INCIDENT_ID_1'] == thin_rx['INCIDENT_ID_2']]
# rename the ID column and the fire acres
thin_rx = thin_rx.rename(columns={"INCIDENT_ID_1": "INCIDENT_ID"})
thin_rx["fire_buffer_acres"] = thin_rx["fire_buffer_acres_1"]

# Recalculate overlap acres and percent
thin_rx["treatment_acres"] = thin_rx.geometry.area / 4046.86
thin_rx["fire_pct_treated"] = (thin_rx["treatment_acres"] / thin_rx["fire_buffer_acres"]) * 100

# Label new treatment type
thin_rx["treatment_type"] = "Thin + Rx Fire"

# Keep consistent schema with main table
thin_rx = thin_rx[[
    "INCIDENT_ID","treatment_type","treatment_acres",
    "fire_buffer_acres","fire_pct_treated","geometry"
]]
thin_rx.head()

,INCIDENT_ID,treatment_type,treatment_acres,fire_buffer_acres,fire_pct_treated,geometry
0,2018_9252264_BLACK CREEK ISLAND,Thin + Rx Fire,0.713540,875.534760,0.081498,"POLYGON ((-9459999.577 3519334.52, -9460005.05..."
1,2020_11831764_BAYTREE,Thin + Rx Fire,18.884879,4010.680685,0.470865,"POLYGON ((-9404094.752 3525591.912, -9404101.7..."
3,2016_4424590_ROCK POND,Thin + Rx Fire,966.979984,5585.863166,17.311201,GEOMETRYCOLLECTION (POLYGON ((-9457060.213 352...
5,2023_16041718_ROCK POND,Thin + Rx Fire,300.480373,7908.287208,3.799563,"MULTIPOLYGON (((-9454898.774 3529377.597, -945..."
11,2023_16041817_112,Thin + Rx Fire,254.562597,9773.746003,2.604555,"MULTIPOLYGON (((-9454857.156 3529388.053, -945..."


In [19]:
# merge the thin+rx fire back
flat_c_fire_int = pd.concat([flat_c_fire_, thin_rx], ignore_index=True)
flat_c_fire_int['treatment_type'].unique()

array(['Broadcast Burn', 'Lop and Scatter', 'Hand Pile Burn',
       'Mastication', 'Machine Pile Burn', 'Machine Pile',
       'Biomass Removal', 'Fire Use', 'Mechanical Thin', 'Manual Thin',
       'Jackpot Burn', 'Hand Pile', 'Thin + Rx Fire'], dtype=object)

In [20]:
# save this long format table
out_fp = os.path.join(projdir, 'data/tabular/MTBS_TWIG_summary_long-format.csv')
flat_c_fire_int.to_csv(out_fp, index=False)
print(f"Saved to: {out_fp}")

Saved to: /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/tabular/MTBS_TWIG_summary_long-format.csv


In [21]:
# make the table wide to easily join with model data
flat_c_fire_w = (
    flat_c_fire_int
    .pivot_table(
        index="INCIDENT_ID",
        columns="treatment_type",
        values="fire_pct_treated",   # proportion of fire treated
        aggfunc="sum",               # if multiple rows per fire/type, sum them
        fill_value=0
    )
    .reset_index()
)
# merge back the other attributes
print(len(flat_c_fire_int))
print(len(flat_c_fire_w))
flat_c_fire_w.head(10)

4516
1594


treatment_type,INCIDENT_ID,Biomass Removal,Broadcast Burn,Fire Use,Hand Pile,Hand Pile Burn,Jackpot Burn,Lop and Scatter,Machine Pile,Machine Pile Burn,Manual Thin,Mastication,Mechanical Thin,Thin + Rx Fire
0,2014_1137798_135A,0.000000,57.378310,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
1,2014_1138291_WEST RANGE FIRE,0.107484,0.897918,0.000000,0.234013,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,1.475495,0.897918
2,2014_247609_SODA,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.192950,0.192945,0.000000,0.0,0.000000,0.000000
3,2014_268085_DOUBLE BUNKER,0.000000,20.554704,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
4,2014_296601_GUN,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.372050,0.390232,0.372050,0.0,0.000000,0.000000
5,2014_298492_TOMS CAMP BRANCH,0.000000,29.233573,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
6,2014_316710_HIGH NOON,0.550957,72.566236,0.000000,0.000000,0.0,0.0,0.0,0.713574,0.000000,0.606248,0.0,5.772003,6.225182
7,2014_317945_GORDON FIRE,0.000000,26.679291,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
8,2014_340088_RICHARD #3,0.582477,72.195093,1.291235,0.000000,0.0,0.0,0.0,0.582477,0.000000,0.000000,0.0,0.000000,0.000000
9,2014_358305_WINDMILL,0.000000,7.725252,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000


In [23]:
# check on the cameron peak fire as an example
flat_c_fire_w[flat_c_fire_w['INCIDENT_ID']=='2020_11856938_CAMERON PEAK']

treatment_type,INCIDENT_ID,Biomass Removal,Broadcast Burn,Fire Use,Hand Pile,Hand Pile Burn,Jackpot Burn,Lop and Scatter,Machine Pile,Machine Pile Burn,Manual Thin,Mastication,Mechanical Thin,Thin + Rx Fire
1017,2020_11856938_CAMERON PEAK,5.063942,0.977068,0.0,0.0,0.0,0.0,0.013526,0.819024,1.199568,0.348244,0.0,0.307211,0.074371


### Previous wildfire burned area

In [24]:
fp = os.path.join(projdir, 'data/spatial/treatments/previous_burns_1km.gpkg')
fires = gpd.read_file(fp)
fires.head()

,fireid,dataset,agency,name,date,lat,lon,poly_area_ha,burn_area_ha,MTBS_name,...,object_ID_usgs,object_ID_iafph,cause_human_or_natural,cause_specific,perimeter_type,fire_year,poly_area_ac,index_right,INCIDENT_ID,geometry
0,20160516_313500_1111100,mtbs,N/A,LA SIERRA,2016-05-16,31.350,-111.110,4466.1147,4069.2573,LA SIERRA,...,82737,N/A,Human,N/A,MTBS,2016.0,11035.992729,465,2022_14492599_TONTO CANYON,"MULTIPOLYGON (((-12373801.575 3681156.062, -12..."
1,20160601_314310_1112180,mtbs,N/A,MULE RIDGE,2016-06-01,31.431,-111.218,3487.9886,3227.5779,MULE RIDGE,...,82738,N/A,Human,N/A,MTBS,2016.0,8618.994230,465,2022_14492599_TONTO CANYON,"MULTIPOLYGON (((-12375530.227 3688485.205, -12..."
2,20210620_313550_1111430,mtbs,N/A,ALAMO,2021-06-20,31.355,-111.143,4240.2999,3457.6177,ALAMO,...,N/A,28351,N/A,N/A,MTBS,2021.0,10477.993068,465,2022_14492599_TONTO CANYON,"MULTIPOLYGON (((-12376879.553 3676338.142, -12..."
3,20090325_314690_1111570,mtbs,N/A,MONTANA,2009-03-25,31.469,-111.157,1039.6383,923.6694,MONTANA,...,81482,N/A,Human,Arson/incendiarism,MTBS,2009.0,2568.998221,470,2016_4300579_MULE RIDGE,"MULTIPOLYGON (((-12372282.806 3694162.916, -12..."
4,20110530_315410_1111580,mtbs,N/A,MURPHY COMPLEX,2011-05-30,31.541,-111.158,26065.0159,22843.1549,MURPHY COMPLEX,...,N/A,N/A,Human,N/A,MTBS,2011.0,64407.957540,470,2016_4300579_MULE RIDGE,"MULTIPOLYGON (((-12367999.953 3711079.15, -123..."


In [26]:
# --- Dissolve by Event ID
fires_flat = fires.dissolve(by=['INCIDENT_ID']).reset_index()
fires_union = gpd.GeoDataFrame(geometry=[fires_flat.union_all()], crs=fires_flat.crs)  # dissolve into one shape

# --- Intersect with MTBS perimeters (buffer)
intersections = gpd.overlay(buffer[['INCIDENT_ID','geometry']], fires_union, how="intersection")
intersections['overlap_acres'] = intersections.geometry.area / 4046.86

# --- Summarize
summary = intersections.groupby("INCIDENT_ID", as_index=False)['overlap_acres'].sum()
summary = summary.merge(buffer[['INCIDENT_ID','fire_buffer_acres']], on='INCIDENT_ID', how='left')
summary['Wildfire'] = summary['overlap_acres'] / summary['fire_buffer_acres'] * 100

pd.set_option("display.float_format", "{:,.4f}".format)
print(summary['Wildfire'].describe())
summary.head()

count   1,026.0000
mean       45.4075
std        32.8307
min         0.0001
25%        13.4129
50%        44.1495
75%        72.3292
max       100.0000
Name: Wildfire, dtype: float64


,INCIDENT_ID,overlap_acres,fire_buffer_acres,Wildfire
0,2014_1075636_DOG ROCK,"2,704.6836","2,704.6836",100.0000
1,2014_247609_SODA,"7,232.6007","7,293.6052",99.1636
2,2014_249140_COLBY,"3,362.4742","8,537.3786",39.3853
3,2014_366230_BASIN,"10,569.3507","17,829.3203",59.2807
4,2014_393350_SIGNAL,"10,342.5501","15,914.5963",64.9878


In [28]:
# merge with the treatment data
trts_w_fire = pd.merge(flat_c_fire_w, summary[['INCIDENT_ID','Wildfire']], on=["INCIDENT_ID"], how="left")
trts_w_fire['Wildfire'] = trts_w_fire['Wildfire'].fillna(0)
trts_w_fire.head()

,INCIDENT_ID,Biomass Removal,Broadcast Burn,Fire Use,Hand Pile,Hand Pile Burn,Jackpot Burn,Lop and Scatter,Machine Pile,Machine Pile Burn,Manual Thin,Mastication,Mechanical Thin,Thin + Rx Fire,Wildfire
0,2014_1137798_135A,0.0000,57.3783,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,2014_1138291_WEST RANGE FIRE,0.1075,0.8979,0.0000,0.2340,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.4755,0.8979,0.0000
2,2014_247609_SODA,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1930,0.1929,0.0000,0.0000,0.0000,0.0000,99.1636
3,2014_268085_DOUBLE BUNKER,0.0000,20.5547,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,2014_296601_GUN,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3720,0.3902,0.3720,0.0000,0.0000,0.0000,0.0000


In [29]:
len(trts_w_fire)

1594

In [30]:
# save the wide format table (just treatment percentages)
out_fp = os.path.join(projdir, 'data/tabular/TWIG_summary_wide-format.csv')
trts_w_fire.to_csv(out_fp, index=False)
print(f"Saved to: {out_fp}")

Saved to: /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/tabular/TWIG_summary_wide-format.csv
